# Kueue-Managed TransformersTrainer: Fine-Tuning Example

This notebook demonstrates a configurable distributed fine-tuning workflow on Red Hat OpenShift AI using Kubeflow Trainer's `TransformersTrainer` and Kueue.

## What it does

1. Downloads a base model and a small subset of the Stanford Alpaca instruction dataset to a shared PVC.
2. Tokenizes the data.
3. Submits a two-node training job through `TransformersTrainer`.
4. Optionally routes the job through a Kueue `LocalQueue`.
5. Stores checkpoints and the final model on the shared PVC.

## Default demo configuration

- **Base model:** `Qwen/Qwen2.5-1.5B-Instruct`
- **Dataset:** `tatsu-lab/alpaca`, first 500 training examples
- **Training:** one epoch, BF16, one GPU per node, two nodes
- **Storage:** a shared PVC with ReadWriteMany (RWX) access
- **Checkpointing:** periodic checkpoints and checkpoint-on-termination, subject to the installed SDK/runtime support

This is a demonstration configuration, not a production training recipe. Validate model/dataset terms, hardware capacity, and the installed OpenShift AI/Kubeflow Trainer versions before use.

## Prerequisites

- OpenShift AI with Kubeflow Trainer v2 and Kueue enabled.
- A Python 3.12+ workbench with the required packages.
- A shared PVC accessible from the workbench and training pods (RWX is recommended for this distributed example).
- Permission to create and inspect training jobs in the target namespace.
- A LocalQueue in the target namespace if Kueue routing is enabled.
- Network access to download the model and dataset during preparation, unless they are already cached on the PVC.

## Security

Do not store API tokens in notebook cells or commit them to Git. Configure `OPENSHIFT_API_URL` and `NOTEBOOK_USER_TOKEN` through the workbench environment or an approved secret mechanism. TLS certificate verification is enabled by default.


## Setup and Imports

Install the Kubeflow SDK and required packages.

In [ ]:
%pip install --upgrade kubeflow datasets transformers accelerate huggingface_hub


In [ ]:
import os
from pathlib import Path

import kubeflow
import torch
from datasets import load_dataset, load_from_disk
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.options import Labels
from kubeflow.trainer.rhai import TransformersTrainer
from kubernetes import client as k8s
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Kubeflow SDK version: {kubeflow.__version__}")
print(f"CUDA available in workbench: {torch.cuda.is_available()}")


## Configuration

The notebook reads cluster authentication and workload settings from environment variables.

Required:
- `OPENSHIFT_API_URL`
- `NOTEBOOK_USER_TOKEN`

Optional:
- `PVC_NAME` (default: `shared`)
- `MODEL_NAME` (default: `Qwen/Qwen2.5-1.5B-Instruct`)
- `KUEUE_QUEUE_NAME` (default: empty; Kueue label is omitted if empty)

The workbench must have the PVC mounted under `/opt/app-root/src/<PVC_NAME>`. Training pods access the same PVC through the SDK's `pvc://` output directory.


In [ ]:
# Authentication: set these as workbench environment variables; never paste tokens here.
api_server = os.getenv("OPENSHIFT_API_URL", "").strip()
token = os.getenv("NOTEBOOK_USER_TOKEN", "").strip()

if not api_server or not token:
    raise RuntimeError(
        "Set OPENSHIFT_API_URL and NOTEBOOK_USER_TOKEN in the workbench environment "
        "before creating the TrainerClient."
    )

configuration = k8s.Configuration()
configuration.host = api_server
configuration.verify_ssl = True
configuration.api_key = {"authorization": f"Bearer {token}"}

# Shared PVC name and paths. Override these environment variables for your cluster.
PVC_NAME = os.getenv("PVC_NAME", "shared")
MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen2.5-1.5B-Instruct")
KUEUE_QUEUE_NAME = os.getenv("KUEUE_QUEUE_NAME", "").strip()

MODEL_DIR_NAME = os.getenv("MODEL_DIR_NAME", "qwen2.5-1.5b-instruct")
DATA_DIR_NAME = os.getenv("DATA_DIR_NAME", "alpaca_processed")
CHECKPOINT_DIR_NAME = os.getenv("CHECKPOINT_DIR_NAME", "kueue-transformers-trainer")

NOTEBOOK_SHARED_PATH = f"/opt/app-root/src/{PVC_NAME}"
NOTEBOOK_MODEL_PATH = f"{NOTEBOOK_SHARED_PATH}/models/{MODEL_DIR_NAME}"
NOTEBOOK_DATA_PATH = f"{NOTEBOOK_SHARED_PATH}/data/{DATA_DIR_NAME}"
NOTEBOOK_CHECKPOINTS_PATH = (
    f"{NOTEBOOK_SHARED_PATH}/checkpoints/{CHECKPOINT_DIR_NAME}"
)
SDK_MOUNT_PATH = "/mnt/kubeflow-checkpoints"

if not os.path.isdir(NOTEBOOK_SHARED_PATH):
    raise FileNotFoundError(
        f"PVC mount not found at {NOTEBOOK_SHARED_PATH}. "
        "Check PVC_NAME and the PVC attached to the workbench."
    )

print(f"Model: {MODEL_NAME}")
print(f"PVC: {PVC_NAME}")
print(f"Workbench PVC mount: {NOTEBOOK_SHARED_PATH}")
print(f"Training pod PVC mount: {SDK_MOUNT_PATH}")
print(f"Kueue LocalQueue: {KUEUE_QUEUE_NAME or 'disabled'}")


## Download Model and Dataset to Shared PVC

Before submitting the training job, we pre-download the model and dataset to the shared PVC. This ensures:
- **Offline Training:** Training pods don't need internet access during training
- **Faster Startup:** No download delays when training pods start
- **Consistency:** All nodes use the same model weights and data

In [ ]:
# Download the base model to the shared PVC once.
if os.path.isdir(NOTEBOOK_MODEL_PATH) and os.listdir(NOTEBOOK_MODEL_PATH):
    print(f"Model already exists at {NOTEBOOK_MODEL_PATH}")
else:
    os.makedirs(NOTEBOOK_MODEL_PATH, exist_ok=True)
    print(f"Downloading {MODEL_NAME} to shared PVC...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    tokenizer.save_pretrained(NOTEBOOK_MODEL_PATH)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
    )
    model.save_pretrained(NOTEBOOK_MODEL_PATH, safe_serialization=True)
    del model
    print(f"Model saved to {NOTEBOOK_MODEL_PATH}")


In [ ]:
# Download and tokenize a small dataset subset for the demo.
if os.path.isdir(NOTEBOOK_DATA_PATH) and os.listdir(NOTEBOOK_DATA_PATH):
    print(f"Processed dataset already exists at {NOTEBOOK_DATA_PATH}")
else:
    os.makedirs(NOTEBOOK_DATA_PATH, exist_ok=True)
    dataset = load_dataset("tatsu-lab/alpaca", split="train[:500]")
    tokenizer = AutoTokenizer.from_pretrained(NOTEBOOK_MODEL_PATH, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def format_instruction(example):
        if example.get("input"):
            text = (
                f"### Instruction:\n{example['instruction']}\n\n"
                f"### Input:\n{example['input']}\n\n"
                f"### Response:\n{example['output']}"
            )
        else:
            text = (
                f"### Instruction:\n{example['instruction']}\n\n"
                f"### Response:\n{example['output']}"
            )
        return {"text": text}

    dataset = dataset.map(format_instruction, remove_columns=dataset.column_names)

    def tokenize_function(examples):
        tokens = tokenizer(
            examples["text"],
            padding="max_length",
            truncation=True,
            max_length=512,
        )
        tokens["labels"] = [ids.copy() for ids in tokens["input_ids"]]
        return tokens

    tokenized_dataset = dataset.map(
        tokenize_function, batched=True, remove_columns=["text"]
    )
    tokenized_dataset.save_to_disk(NOTEBOOK_DATA_PATH)
    print(f"Processed dataset saved to {NOTEBOOK_DATA_PATH}")


## Define the Training Function

The training function runs inside each training pod as a distributed PyTorch process. TransformersTrainer serializes this function and executes it via `torchrun` on each node.

### Training Configuration

| Parameter | Value | Description |
|-----------|-------|-------------|
| `num_train_epochs` | 1 | Complete one pass through the dataset |
| `per_device_train_batch_size` | 2 | Samples per GPU per step |
| `gradient_accumulation_steps` | 4 | Effective batch size = 2 × 4 × 2 nodes = 16 |
| `learning_rate` | 2e-5 | Standard fine-tuning rate |
| `save_steps` | 20 | Checkpoint every 20 steps |
| `bf16` | True | Use bfloat16 mixed precision |

### Key Points

- **Supported Trainers:** Use `transformers.Trainer` or `trl.SFTTrainer` - both are auto-instrumented
- **No Manual Setup:** Progress tracking callback is injected automatically
- **Local Files Only:** Model and data are loaded from the mounted PVC (no network access needed)

In [ ]:
def train_func():
    """Supervised fine-tuning function executed by TransformersTrainer on each node."""
    import os
    import torch
    from datasets import load_from_disk
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        DataCollatorForLanguageModeling,
        Trainer,
        TrainingArguments,
    )

    rank = int(os.environ.get("RANK", "0"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    pvc_mount = "/mnt/kubeflow-checkpoints"
    model_path = os.path.join(pvc_mount, "models", os.environ["MODEL_DIR_NAME"])
    data_path = os.path.join(pvc_mount, "data", os.environ["DATA_DIR_NAME"])

    print(f"Starting training on rank {rank}")
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        print(f"GPU: {torch.cuda.get_device_name(local_rank)}")

    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        local_files_only=True,
    )
    tokenized_dataset = load_from_disk(data_path)
    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    training_args = TrainingArguments(
        output_dir="/tmp/output",  # TransformersTrainer configures the PVC output path.
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-5,
        logging_steps=5,
        report_to="none",
        bf16=True,
        ddp_find_unused_parameters=False,
    )
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
    )
    trainer.train()

    if rank == 0:
        final_path = os.path.join(trainer.args.output_dir, "final")
        os.makedirs(final_path, exist_ok=True)
        trainer.save_model(final_path)
        tokenizer.save_pretrained(final_path)
        print(f"Final model saved to {final_path}")

    print(f"Training complete on rank {rank}")

print("Training function defined")


## Create the Trainer Client

Initialize the TrainerClient with authentication configuration.

In [ ]:
api_client = k8s.ApiClient(configuration)
backend_config = KubernetesBackendConfig(client_configuration=api_client.configuration)
client = TrainerClient(backend_config)
runtime = client.backend.get_runtime("torch-distributed")

print("TrainerClient created")
print(f"Using runtime: {runtime.name}")


## Submit the Training Job

The example below requests two training nodes, each with one NVIDIA GPU, 4 CPU cores, and 16 GiB memory. Adjust these values to match your cluster quota and GPU capacity.

If `KUEUE_QUEUE_NAME` is non-empty, the job receives the Kueue LocalQueue label. Check that the LocalQueue exists in the namespace where the TrainJob is created.

The output URI uses the configured PVC name, so the SDK can mount the same PVC in the training pods and store checkpoints there.


In [ ]:
from kubeflow.trainer.rhai.transformers import PeriodicCheckpointConfig

checkpoint_config = PeriodicCheckpointConfig(
    save_strategy="steps",
    save_steps=20,
    save_total_limit=2,
)

trainer = TransformersTrainer(
    func=train_func,
    num_nodes=2,
    resources_per_node={
        "nvidia.com/gpu": 1,
        "cpu": "4",
        "memory": "16Gi",
    },
    env={
        "HF_HUB_OFFLINE": "1",
        "TRANSFORMERS_OFFLINE": "1",
        "MODEL_DIR_NAME": MODEL_DIR_NAME,
        "DATA_DIR_NAME": DATA_DIR_NAME,
        "NCCL_IB_DISABLE": "1",
        "NCCL_P2P_DISABLE": "1",
        "NCCL_DEBUG": "INFO",
    },
    enable_jit_checkpoint=True,
    periodic_checkpoint_config=checkpoint_config,
    output_dir=f"pvc://{PVC_NAME}/checkpoints/{CHECKPOINT_DIR_NAME}",
)

kueue_options = (
    [Labels(labels={"kueue.x-k8s.io/queue-name": KUEUE_QUEUE_NAME})]
    if KUEUE_QUEUE_NAME
    else None
)

job_name = client.train(
    trainer=trainer,
    runtime=runtime,
    options=kueue_options,
)
print(f"Training job submitted: {job_name}")
print(f"Progress tracking is provided by the installed TransformersTrainer SDK.")
print(f"Checkpoints should be available at: {NOTEBOOK_CHECKPOINTS_PATH}")


## Follow Job Logs

Let's fetch our job logs to make sure training is going as expected. The logs will stream in real-time as the training progresses.

In [ ]:
# Stream logs while the job is running. Interrupt this cell to stop following logs.
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end="")


## Get Job Status

Check the final status of the training job after completion.

In [ ]:
job = client.get_job(job_name)
print("TrainJob status:")
print(f"Name: {job.name}")
print(f"Status: {job.status}")
print(f"Created: {job.creation_timestamp}")
print(f"Nodes: {job.num_nodes}")
print(f"Runtime: {job.runtime.name}")
if job.steps:
    for step in job.steps:
        print(f"- {step.name}: {step.status}")


## Check Training Progress

View training progress directly in the OpenShift AI Dashboard (Training Jobs).

## Test the Trained Model (Optional)

After training completes, you can load the fine-tuned model from the checkpoint saved on the shared PVC.

### Checkpoint Structure

The training function saves checkpoints with this structure:
```
/opt/app-root/src/shared/checkpoints/kueue-transformers-trainer/
├── checkpoint-<step>/  # Intermediate checkpoints (saved every save_steps)
├── checkpoint-<N>/    # Checkpoint at final step (N = last step number)
└── final/             # Final merged model ready for inference
```

### Testing the Model

We'll load the fine-tuned model and test it with an instruction prompt using the same format as the training data:

```
### Instruction:
<your instruction here>

### Response:
<model generates response>
```

In [ ]:
def find_most_recent_checkpoint(output_dir):
    """Find the most recently created checkpoint directory."""
    if not os.path.exists(output_dir):
        raise FileNotFoundError(f"Output directory not found: {output_dir}")

    checkpoint_dirs = [
        os.path.join(output_dir, d)
        for d in os.listdir(output_dir)
        if os.path.isdir(os.path.join(output_dir, d))
        and (d.startswith("checkpoint-") or d == "final")
    ]

    if not checkpoint_dirs:
        raise FileNotFoundError(f"No checkpoints found in {output_dir}")

    # Prefer 'final' if it exists
    final_path = os.path.join(output_dir, "final")
    if os.path.exists(final_path):
        return final_path

    return max(checkpoint_dirs, key=os.path.getctime)


print("✅ Checkpoint utility defined")

In [ ]:
# Optional: load and test the final model after the training job completes.
final_checkpoint = os.path.join(NOTEBOOK_CHECKPOINTS_PATH, "final")
if not os.path.isdir(final_checkpoint):
    raise FileNotFoundError(
        f"Final model not found at {final_checkpoint}. Confirm that training completed successfully."
    )

trained_tokenizer = AutoTokenizer.from_pretrained(final_checkpoint)
trained_model = AutoModelForCausalLM.from_pretrained(
    final_checkpoint,
    torch_dtype=torch.bfloat16,
    device_map="auto" if torch.cuda.is_available() else "cpu",
)

test_prompt = "### Instruction:\nExplain what machine learning is in one sentence.\n\n### Response:"
inputs = trained_tokenizer(test_prompt, return_tensors="pt").to(trained_model.device)
with torch.no_grad():
    outputs = trained_model.generate(**inputs, max_new_tokens=100, do_sample=False)

print(trained_tokenizer.decode(outputs[0], skip_special_tokens=True))


## Optional Cleanup

Deleting a TrainJob may remove its Kubernetes job resources. Do not run the next cell until you have confirmed that the job and its logs are no longer needed. Checkpoints stored on the PVC are separate and should be reviewed before deleting any files.


In [ ]:
# Optional: delete the submitted TrainJob only after reviewing its status and retaining needed logs.
# Uncomment the next line when you explicitly want to delete this job.
# client.delete_job(name=job_name)
print(f"Cleanup not performed. TrainJob: {job_name}")


In [ ]:
import gc

# Clear CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

gc.collect()
print("✅ Resources freed, CUDA cache cleared")

## Summary

This notebook provides a configurable example of distributed supervised fine-tuning on OpenShift AI.

It demonstrates:
- Preparing a model and a small instruction dataset on a shared PVC.
- Submitting a distributed `TransformersTrainer` job.
- Optionally routing the job through Kueue.
- Inspecting logs and job status.
- Saving and optionally testing the final model.

Before using this workflow for a real experiment, record the model revision, dataset revision, environment versions, GPU/CPU/memory requests, training hyperparameters, and resulting metrics. The default settings are a demonstration and should be tuned and validated for the target environment.

### Important

- Never commit API tokens or other credentials.
- Use TLS verification and the cluster's approved certificate configuration.
- Confirm that the PVC is shared by the workbench and training pods.
- Confirm the Kueue LocalQueue and resource quota before submitting.
- This notebook has been cleaned of saved outputs and has not been executed against a cluster.
